# Result · Every number quoted in the paper, and their provenance

**In the paper ([arXiv:2610.10899](https://arxiv.org/abs/2610.10899)):** every number. The provenance check runs on `paper/main.tex` and on the LaTeX source posted to arXiv.

Recomputes every evaluation (77 GPU jobs, about 17 A100-hours), rebuilds all tables and the figure, compares all
quoted numbers (`analysis/out/paper_numbers.json`) with the committed values, and checks that every number in
`paper/main.tex` and in the version posted to arXiv traces to a generated file.

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters (also settable through environment variables for non-interactive runs)
# SOURCE = "recompute": re-run the evaluations behind this output from the released checkpoints (GPU), then aggregate.
# SOURCE = "released" : aggregate the released outputs of the original runs (CPU, seconds). This rewrites the
#                       committed analysis/out/ and paper/ files, which should come out identical.
SOURCE = os.environ.get("NB_SOURCE", "recompute")
# RUN = "slurm": one GPU job per run (sbatch), this kernel waits; "local": run in this kernel (needs a GPU).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['ablation', 'bench', 'e1', 'e1_attacker_seeds', 'e2', 'e3_eval', 'e3_eval_r2', 'hnp', 'multispace']]
print(len(KEYS), "runs:", ", ".join(KEYS))

77 runs: e2_R0_s0, e2_R0_s1, e2_R0_s2, hnp_s0, hnp_s1, hnp_s2, abl_tau0.1_s0, abl_margin0.1_s0, abl_headonly_s0, abl_tau0.5_s0, abl_margin1.0_s0, abl_prefinput_s0, e1_R0_s0, e1_R3_s0, e1_R0_s1, e1_R3_s1, e1_R0_s2, e1_R3_s2, e1_baseline_noise_s0, e1_baseline_pgd_s0, e1_baseline_random_s0, e1_baseline_noise_s1, e1_baseline_pgd_s1, e1_baseline_random_s1, e1_baseline_noise_s2, e1_baseline_pgd_s2, e1_baseline_random_s2, e1_R1_s0, e1_R2_s0, e1_R1_s1, e1_R2_s1, e1_R1_s2, e1_R2_s2, e2_R1_s0, e2_R2_s0, e2_R3_s0, e2_baseline_noise_s0, e2_baseline_pgd_s0, e2_baseline_random_s0, e2_R1_s1, e2_R2_s1, e2_R3_s1, e2_baseline_noise_s1, e2_baseline_pgd_s1, e2_baseline_random_s1, e2_R1_s2, e2_R2_s2, e2_R3_s2, e2_baseline_noise_s2, e2_baseline_pgd_s2, e2_baseline_random_s2, e2_abl_tau0p1_s0_s0, e2_abl_margin0p1_s0_s0, e2_abl_headonly_s0_s0, e2_abl_margin1p0_s0_s0, e2_abl_prefinput_s0_s0, e3_eval_s0, e3_eval_s1, e3_eval_s2, e2_abl_tau0p5_s0_s0, ms_s0, ms_s1, ms_s2, e1_R3ms_s0, e2_R3ms_s0, e1_R3ms_s1, e2_R3m

In [4]:
if SOURCE == "recompute":
    repro.run_keys(KEYS, "recompute", RUN)  # skips runs whose recomputed output already exists
dirs = repro.run_analysis(SOURCE)  # whole analysis chain (seconds); falls back to released outputs where needed
OUT, PAPER = dirs["out"], dirs["paper"]
COMMITTED_OUT, COMMITTED_PAPER = ROOT / "analysis" / "out", ROOT / "paper"

[analysis] runs=/fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/runs
[analysis] out=/users/PAS2301/alialavi/projects/audio_assessment_hackability/analysis/out
[analysis] paper=/users/PAS2301/alialavi/projects/audio_assessment_hackability/paper


In [5]:
# Where this notebook's inputs came from: recomputed from checkpoints, or released outputs of the original runs.
used = repro.load_registry(repro.results_registry(SOURCE))
pd.Series({k: used[k].get("source", "released") for k in KEYS}, name="source").value_counts()

source
released    77
Name: count, dtype: int64

In [6]:
# Numbers the paper quotes from this output: committed value vs this notebook's value.
cmp = repro.compare_numbers(OUT / "paper_numbers.json", prefixes=())
print(f"{len(cmp)} numbers; max |diff| = {cmp.abs_diff.max():.4f}; identical: {(cmp.abs_diff == 0).sum()}")
cmp.head(20)

1902 numbers; max |diff| = 0.0000; identical: 1902


,paper,reproduced,abs_diff
all_input_score_mean,3.200,3.200,0.0
asr_R0_grpo_dac,0.120,0.120,0.0
asr_R0_grpo_dac_2dp,0.120,0.120,0.0
asr_R0_grpo_dac_sd,0.192,0.192,0.0
asr_R0_grpo_dac_seed0,0.015,0.015,0.0
asr_R0_grpo_dac_seed1,0.341,0.341,0.0
asr_R0_grpo_dac_seed2,0.004,0.004,0.0
asr_R0_grpo_encodec,0.181,0.181,0.0
asr_R0_grpo_encodec_2dp,0.180,0.180,0.0
asr_R0_grpo_encodec_sd,0.140,0.140,0.0


In [7]:
import subprocess
r = subprocess.run([sys.executable, "scripts/number_provenance_linter.py",
                    "paper/main.tex", "--artifacts", str(OUT)], capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:], "exit", r.returncode)

Checked paper/main.tex against 12338 numbers in /users/PAS2301/alialavi/projects/audio_assessment_hackability/analysis/out
PASS: every paper number traces to a generated artifact.
  exit 0


The same check on the LaTeX source posted to arXiv (downloaded from `arxiv.org/src`), so every number a reader
sees there is covered too.

In [8]:
arxiv = repro.fetch_arxiv_source()  # latest version; fetch_arxiv_source("v1") for the first
r = subprocess.run([sys.executable, "scripts/number_provenance_linter.py", str(arxiv / "main.tex"),
                    "--artifacts", str(OUT)], capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:], "exit", r.returncode)

Checked /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/arxiv_2610.10899_latest/main.tex against 12338 numbers in /users/PAS2301/alialavi/projects/audio_assessment_hackability/analysis/out
PASS: every paper number traces to a generated artifact.
  exit 0
